<a href="https://colab.research.google.com/github/TaherBenAfia/Fly2/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**In words.** A page earns a spot in the queue when demand is present and something is off: it is *observed* declining, stale, under-converting, or under-engaging. The rank is the advice — the top of the queue is where a small edit is most likely to hold traffic, and `monitor` is the answer for everything else.

Priority, first to last:

1. `refresh_and_review_ctr` — declining, still visible near page 1, low CTR. Review title / meta / intent for the query it is losing. Highest leverage: the click gap, not the visibility gap.
2. `refresh_and_review_engagement` — declining, visible, low engagement. Review body / UX against what the query wants.
3. `refresh` — declining or at-risk, but no single confirmed lever yet; a general refresh, re-check next window.
4. `expand_and_refresh` — thin but visible page (short content with demand); needs depth, not a copy edit.
5. `monitor` — watch, don't touch.

**Evidence language.** Reason codes use the words we can carry: `observed` (30-day trend state), `directional` (model decline risk), `decision-support` (the recommended action). The model numbers below are the honest, client-grouped ones — measured on held-out client groups, not fitted on packed rows.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

if "google.colab" in sys.modules:
    os.chdir("flyrank-ml-internship-starter" if os.path.isdir("flyrank-ml-internship-starter") else ".")
else:
    for _ in range(3):
        if os.path.isdir("data/raw"):
            break
        os.chdir("..")
print("cwd:", os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
y = df["is_declining_label"].values
print(df.shape, "| declining base rate:", round(y.mean(), 3))

# Leakage-safe features (same as w05/w06): drop the two 30-day windows that overlap the label.
df["impressions_earliest_slice"] = (df["impressions_90d"] - df["impressions_last_30d"] - df["impressions_prev_30d"]).clip(lower=0)
df["log_impressions_earliest_slice"] = np.log1p(df["impressions_earliest_slice"])
df.loc[df["avg_position"] == 0, "avg_position"] = np.nan  # 0 = "no data"

numeric_features = ["search_volume", "competition", "cpc", "word_count", "char_count",
                    "log_impressions_earliest_slice", "content_age_days", "days_since_last_update",
                    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
categorical_features = ["competition_level", "content_type", "main_intent", "age_tier",
                        "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]

X_num = df[numeric_features].fillna(0)
X_cat = pd.get_dummies(df[categorical_features].fillna("unknown"))
X = pd.concat([X_num, X_cat], axis=1)


def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


# Honest out-of-fold decline probability: GroupKFold by client, whole client-groups held out.
oof = np.zeros(len(df))
aucs, p20s, p50s = [], [], []
gkf = GroupKFold(n_splits=5)
for train_idx, test_idx in gkf.split(X, y, groups=df["client_id"]):
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X.iloc[train_idx], y[train_idx])
    proba = rf.predict_proba(X.iloc[test_idx])[:, 1]
    oof[test_idx] = proba
    aucs.append(roc_auc_score(y[test_idx], proba))
    p20s.append(precision_at_k(proba, y[test_idx], 20))
    p50s.append(precision_at_k(proba, y[test_idx], 50))

df["model_decline_prob"] = oof
print("honest GroupKFold by client -> AUC %.3f | P@20 %.2f | P@50 %.2f | base rate %.3f"
      % (np.mean(aucs), np.mean(p20s), np.mean(p50s), y.mean()))

# Transparent rule flags -> the reason codes the queue shows a reviewer.
visible = df["impressions_90d"] >= 500
df["stale_visible"] = ((df["days_since_last_update"] >= 180) & visible).astype(int)
df["declining_with_demand"] = ((df["trend_direction"].str.lower() == "down") & (df["impressions_90d"] >= 100)).astype(int)
df["thin_visible"] = ((df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)).astype(int)
df["page_one_decay"] = ((df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)).astype(int)
df["low_ctr_visible"] = (visible & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)).astype(int)
df["low_eng_visible"] = ((df["sessions_90d"] >= 30) &
                         (((df["engagement_rate"] > 0) & (df["engagement_rate"] < 30)) |
                          ((df["scroll_rate"] > 0) & (df["scroll_rate"] < 30)))).astype(int)

# Transparent baseline score: demand, decline-with-demand, staleness, under-conversion.
log_imp = np.log1p(df["impressions_90d"])
visibility = (log_imp - log_imp.min()) / (log_imp.max() - log_imp.min())
df["baseline_refresh_score"] = (0.40 * visibility
    + 0.30 * df["declining_with_demand"] * visibility
    + 0.20 * np.clip(df["days_since_last_update"] / 180, 0, 1)
    + 0.10 * ((df["low_ctr_visible"] + df["low_eng_visible"]) / 2)).clip(0, 1)

df["final_refresh_score"] = 100 * (0.70 * df["model_decline_prob"] + 0.30 * df["baseline_refresh_score"]).clip(0, 1)


def reason_codes(row):
    reasons = []
    if row["stale_visible"]:
        reasons.append("stale_visible_page")
    if row["declining_with_demand"]:
        reasons.append("declining_with_demand")
    if row["thin_visible"]:
        reasons.append("thin_visible_page")
    if row["page_one_decay"]:
        reasons.append("page_one_decay_risk")
    if row["low_ctr_visible"]:
        reasons.append("low_ctr_visible_page")
    if row["low_eng_visible"]:
        reasons.append("low_engagement_visible_page")
    if row["model_decline_prob"] >= 0.65:
        reasons.append("model_decline_risk")
    if row["model_decline_prob"] >= 0.5 and row["impressions_90d"] >= 500:
        reasons.append("visible_model_opportunity")
        if row["low_ctr_visible"]:
            reasons.append("ctr_review_candidate")
        if row["low_eng_visible"]:
            reasons.append("engagement_review_candidate")
    return "|".join(reasons) if reasons else "general_refresh_review"


df["final_reason_codes"] = df.apply(reason_codes, axis=1)


def suggested_action(row):
    reasons = set(row["final_reason_codes"].split("|"))
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "ctr_review_candidate" in reasons and ("model_decline_risk" in reasons or "declining_with_demand" in reasons):
        return "refresh_and_review_ctr"
    if "engagement_review_candidate" in reasons and ("model_decline_risk" in reasons or "declining_with_demand" in reasons):
        return "refresh_and_review_engagement"
    if {"model_decline_risk", "declining_with_demand", "stale_visible_page", "visible_model_opportunity"} & reasons:
        return "refresh"
    return "monitor"


df["suggested_action"] = df.apply(suggested_action, axis=1)

high_t = df["final_refresh_score"].quantile(0.8)
med_t = df["final_refresh_score"].quantile(0.5)


def confidence(row):
    if (row["final_refresh_score"] >= high_t and row["impressions_90d"] >= 500
            and row["sessions_90d"] >= 10 and row["model_decline_prob"] >= 0.5):
        return "high"
    if row["final_refresh_score"] >= med_t:
        return "medium"
    return "low"


df["confidence"] = df.apply(confidence, axis=1)

queue = df.sort_values(["final_refresh_score", "impressions_90d", "sessions_90d"],
                       ascending=[False, False, False]).reset_index(drop=True)
queue["final_rank"] = queue.index + 1

print("actions:", df["suggested_action"].value_counts().to_dict())
print("confidence:", df["confidence"].value_counts().to_dict())

cwd: C:\Users\taher\OneDrive\Documents\first_assignment_FLYRANK\Fly2


(30000, 45) | declining base rate: 0.542


honest GroupKFold by client -> AUC 0.650 | P@20 0.76 | P@50 0.73 | base rate 0.542


actions: {'monitor': 11540, 'refresh': 11010, 'refresh_and_review_ctr': 5633, 'refresh_and_review_engagement': 1735, 'expand_and_refresh': 82}
confidence: {'low': 15000, 'medium': 11077, 'high': 3923}


In [ ]:
# Top of the queue: what a reviewer sees on Monday.
show = queue.head(20)[["final_rank", "content_id", "impressions_90d", "sessions_90d",
                       "avg_position", "ctr", "model_decline_prob", "final_refresh_score",
                       "confidence", "suggested_action", "final_reason_codes", "trend_direction"]]
print("top-20 declining rate:", round(queue.head(20)["is_declining_label"].mean(), 3), "(base rate 0.542)")
show

top-20 declining rate: 1.0 (base rate 0.542)


,final_rank,content_id,impressions_90d,sessions_90d,avg_position,ctr,model_decline_prob,final_refresh_score,confidence,suggested_action,final_reason_codes,trend_direction
0,1,content_f988b4cba4ea,16045,240,40.8,0.01,0.817409,77.332209,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down
1,2,content_b60c70399545,20599,112,30.3,0.08,0.798287,76.414589,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down
2,3,content_dc07a16ea110,24620,81,28.0,0.02,0.791185,76.217884,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down
3,4,content_813e88069237,233561,1538,26.2,0.06,0.736103,76.152767,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down
4,5,content_4f5826036689,9761,65,21.9,0.06,0.812013,76.117180,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down
5,6,content_987a0c9b0910,13756,130,27.6,0.04,0.799977,75.852616,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down
6,7,content_2028099fd151,20296,92,23.5,0.09,0.790373,75.835631,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down
7,8,content_47cfaff33c47,20248,593,26.4,0.16,0.787913,75.659435,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down
8,9,content_bbb7b13eac87,17042,53,35.3,0.05,0.791693,75.633615,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down
9,10,content_26aaf62c0a5d,29618,222,35.4,0.07,0.774882,75.388032,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,down


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses this, and for what.** A human content/SEO reviewer at FlyRank, weekly, deciding which pages to look at first and what to check. One decision per page, in order. It is decision-support: it orders the review queue and names the suspicion; it does not write or publish anything.

**Where it stops being valid.**

- One snapshot, one portfolio, one 90-day window: no calendar survived anonymization, so no seasonality can be tested and no forward "what happens next" claim is made.
- No causal claim is possible here: cross-sectional data with no intervention cannot say that refreshing X *will* recover Y.
- Rows without position / keyword / GA4 data get partial reason codes — the model never saw those columns, so their confidence is capped.
- Client history is uneven; the honest precision is an average over client groups, and a single client can dominate the top of the queue (the code below measures both).
- `monitor` means no confirmed lever yet — not a promise of safety.

In [ ]:
# Where the numbers are weaker: count the rows the queue must treat carefully.
print("rows without position data:", int(df["avg_position"].isna().sum()))
print("rows without keyword data:", int(df["search_volume"].isna().sum()))
print("rows with zero GA4 sessions:", int((df["sessions_90d"] == 0).sum()))

top100 = queue.head(100)
print()
print("top-100 rows from the single largest client:", int(top100["client_id"].value_counts().max()))
print("top-100 distinct clients:", int(top100["client_id"].nunique()))
print("top-100 declining rate:", round(float(top100["is_declining_label"].mean()), 3))
print()
print("action x confidence (whole queue):")
print(pd.crosstab(df["suggested_action"], df["confidence"]))

rows without position data: 1205
rows without keyword data: 2468
rows with zero GA4 sessions: 0

top-100 rows from the single largest client: 96
top-100 distinct clients: 3
top-100 declining rate: 1.0

action x confidence (whole queue):
confidence                     high    low  medium
suggested_action                                  
expand_and_refresh                5     41      36
monitor                           0  11226     314
refresh                         524   3666    6820
refresh_and_review_ctr         2090     65    3478
refresh_and_review_engagement  1304      2     429


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**A person must check before acting (on any row they touch):**

1. **Is the trend real?** `down` is only two 30-day windows — confirm no launch, sitemap break, or tracking change explains it.
2. **Is the reason code the actual problem?** `low_ctr_visible_page` on page 1 can be a SERP feature (video carousel, AI overview) stealing the click, not a bad title.
3. **Is the measurement intact?** GA4 can break silently; a `low_engagement_visible_page` flag on a broken tracking tag is noise.
4. **Was the last update logged?** `stale_visible_page` rests on `days_since_last_update` being truthful.
5. **Did the model have the evidence?** If position or keyword data is missing, drop confidence one notch.

**The no-go list (never automated):**

- No automatic publishing or unpublishing of content.
- No deletes or deprecations based on the score.
- No treating `declining_with_demand` as if we had *caused* or could *cause* the decline — it is observed state, not a causal link.
- No running the queue on brand-new pages (< 90 days), rows without position data, or clients without GA4.
- No claiming the model "predicted Google's algorithm" — it ranked pages inside one pseudonymized portfolio.

In [ ]:
# Rows that still need a human eye, and why.

# (a) Model says "decline risk" (0.65+) but the observed trend is flat/up:
#     a possible misfire - inspect before trusting the queue placement.
misfire = (queue["model_decline_prob"] >= 0.65) & (queue["trend_direction"] != "down")
# (b) High-risk rows with no position data: the model could not see where the page ranks.
nopos = (queue["model_decline_prob"] >= 0.5) & queue["avg_position"].isna()

print("whole queue -- model says 0.65+ risk while trend is NOT down:", int(misfire.sum()))
print("whole queue -- high-risk rows with no position data:", int(nopos.sum()))
print()

flex = queue[misfire | nopos][["final_rank", "content_id", "model_decline_prob", "trend_direction",
                               "avg_position", "impressions_90d", "sessions_90d", "suggested_action"]]
print("top of that list (what a reviewer inspects on Monday):")
flex.head(4)

whole queue -- model says 0.65+ risk while trend is NOT down: 1461
whole queue -- high-risk rows with no position data: 1

top of that list (what a reviewer inspects on Monday):


,final_rank,content_id,model_decline_prob,trend_direction,avg_position,impressions_90d,sessions_90d,suggested_action
309,310,content_3164f3076003,0.821909,up,16.1,2696,3,refresh_and_review_ctr
327,328,content_1577e55cbe22,0.790070,stable,22.4,21228,447,refresh_and_review_engagement
379,380,content_7852e54e8f47,0.803736,stable,23.1,4105,146,refresh_and_review_engagement
380,381,content_b2ec40b9d68c,0.779701,up,47.3,23457,383,refresh_and_review_engagement


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The queue is a snapshot. Treat it as fresh for about 4 weeks, then re-score. Staleness shows up as:

1. **Score drift** — the median `final_refresh_score` moves without a re-score → the underlying mix changed.
2. **Signal drift** — the position/impression-tier mix shifts (re-platform, relaunch) → the rule assumptions break.
3. **Label drift** — the measured `down` rate moves far from 0.542 → every precision number needs re-reading.
4. **Performance drift** — the honest check is the *lagged* label (the 30–60 day windows behind us): does the top-50 declining rate still sit well above the base rate? When it falls toward base, retrain.

Start-line triggers (tune with the team):

- Re-score every 4 weeks, or the day a client resyncs data.
- Retrain when honest P@50 < ~0.60, or when the top-50 declining rate < base rate + 0.10.
- Alert when one client exceeds ~1/3 of the top-100 — the queue is then that client's backlog, and a per-client cap should kick in.

In [ ]:
base_rate = float(y.mean())
top50_decline = float(queue.head(50)["is_declining_label"].mean())

print("base down rate:", round(base_rate, 3))
print("observed top-50 declining rate (lagged-label proxy):", round(top50_decline, 3))
print("median final_refresh_score:", round(float(df["final_refresh_score"].median()), 1))
print()

checks = {
    "retrain floor: top-50 declining rate >= base + 0.10": top50_decline >= base_rate + 0.10,
    "client cap: no single client > 1/3 of top-100": int(queue.head(100)["client_id"].value_counts().max()) <= 33,
    "position coverage >= 95%": (1 - float(df["avg_position"].isna().mean())) >= 0.95,
}
for label, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + label)

base down rate: 0.542
observed top-50 declining rate (lagged-label proxy): 1.0
median final_refresh_score: 49.0

PASS retrain floor: top-50 declining rate >= base + 0.10
FAIL client cap: no single client > 1/3 of top-100
PASS position coverage >= 95%


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The paper embeds these three files: the full ranked queue (pseudonymized), the metric numbers, and the two simple charts below. No client names, URLs, or private queries anywhere.

In [ ]:
os.makedirs("work/outputs", exist_ok=True)

save_cols = ["final_rank", "content_id", "client_id", "final_refresh_score", "model_decline_prob",
             "baseline_refresh_score", "confidence", "suggested_action", "final_reason_codes",
             "impressions_90d", "sessions_90d", "avg_position", "ctr", "content_age_days",
             "days_since_last_update", "word_count", "trend_direction", "is_declining_label"]
queue[save_cols].to_csv("work/outputs/playbook_refresh_queue.csv", index=False)

metrics = {
    "rows_scored": int(len(df)),
    "declining_base_rate": round(float(y.mean()), 3),
    "honest_groupkfold_auc": round(float(np.mean(aucs)), 3),
    "honest_groupkfold_p20": round(float(np.mean(p20s)), 3),
    "honest_groupkfold_p50": round(float(np.mean(p50s)), 3),
    "top20_declining_rate": round(float(queue.head(20)["is_declining_label"].mean()), 3),
    "action_counts": {str(k): int(v) for k, v in df["suggested_action"].value_counts().items()},
    "confidence_counts": {str(k): int(v) for k, v in df["confidence"].value_counts().items()},
}
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)


def simple_bar_chart(title, labels, values, path, color="#426B69"):
    labels = [str(x) for x in labels]
    values = [float(v) for v in values]
    m = max(max(values, default=1), 1)
    margin_l, margin_r, margin_t, margin_b = 210, 40, 70, 50
    pw, ph = 960 - margin_l - margin_r, 520 - margin_t - margin_b
    gap = 10
    bh = max(14.0, (ph - gap * max(len(values) - 1, 0)) / max(len(values), 1))
    lines = ['<svg xmlns="http://www.w3.org/2000/svg" width="960" height="520" viewBox="0 0 960 520">',
             '<rect width="100%" height="100%" fill="#ffffff"/>',
             f'<text x="480" y="34" text-anchor="middle" font-family="Arial" font-size="24" fill="#16232a">{title}</text>']
    for i, (label, value) in enumerate(zip(labels, values)):
        y = margin_t + i * (bh + gap)
        bw = (value / m) * pw
        lines.append(f'<text x="{margin_l - 12}" y="{y + bh * 0.65:.1f}" text-anchor="end" font-family="Arial" font-size="13" fill="#27343b">{label[:32]}</text>')
        lines.append(f'<rect x="{margin_l}" y="{y:.1f}" width="{bw:.1f}" height="{bh:.1f}" fill="{color}" rx="4"/>')
        lines.append(f'<text x="{margin_l + bw + 8:.1f}" y="{y + bh * 0.65:.1f}" font-family="Arial" font-size="13" fill="#27343b">{value:,.0f}</text>')
    lines.append("</svg>")
    with open(path, "w") as f:
        f.write("\n".join(lines))


simple_bar_chart("Playbook action mix",
                 df["suggested_action"].value_counts().index.tolist(),
                 df["suggested_action"].value_counts().values.tolist(),
                 "work/outputs/playbook_action_mix.svg", color="#426B69")
simple_bar_chart("Playbook confidence mix",
                 ["high", "medium", "low"],
                 [int((df["confidence"] == c).sum()) for c in ["high", "medium", "low"]],
                 "work/outputs/playbook_confidence_mix.svg", color="#6F4E7C")

print(metrics)
print()
print("files in work/outputs:")
for name in sorted(os.listdir("work/outputs")):
    print(" -", name)

{'rows_scored': 30000, 'declining_base_rate': 0.542, 'honest_groupkfold_auc': 0.65, 'honest_groupkfold_p20': 0.76, 'honest_groupkfold_p50': 0.732, 'top20_declining_rate': 1.0, 'action_counts': {'monitor': 11540, 'refresh': 11010, 'refresh_and_review_ctr': 5633, 'refresh_and_review_engagement': 1735, 'expand_and_refresh': 82}, 'confidence_counts': {'low': 15000, 'medium': 11077, 'high': 3923}}

files in work/outputs:
 - baseline_action_score_metrics.json
 - playbook_action_mix.svg
 - playbook_confidence_mix.svg
 - playbook_metrics.json
 - playbook_refresh_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.